# SBAS-интерферометрия пашни Татарстана на части сцены Sentinel-1

Этот ноутбук переделан под ваш сценарий: **обрабатывается не одно поле и не один burst, а часть сцены, задаваемая AOI**. Поэтому здесь используется **поиск и скачивание SLC-сцен**, затем ограничение по `SUBSWATH` и дополнительное пространственное сужение через `compute_reframe(AOI)`.

## Ключевая сезонная логика

- основной сезон для пашни: **сентябрь–ноябрь**;
- **февраль** сохраняется как межгодовой мост;
- декабрь, январь и март по умолчанию не используются;
- сеть SBAS строится из коротких осенних пар и ограниченного числа длинных связей через февраль.

## Что меняется по сравнению с burst-стеком

- выбирается **одна устойчивая сцена/трек** по AOI;
- скачиваются **частичные SLC-сцены** через `ASF.download(..., subswaths=SUBSWATH)`;
- геометрия затем дополнительно режется по `AOI` с помощью `compute_reframe(AOI)`.

> Ноутбук опирается не только на опубликованные примеры, но и на **текущие сигнатуры методов в исходном коде** `ASF.py`, `S1.py`, `Stack_reframe.py`, `Stack_phasediff.py`, `Stack_unwrap.py`, `Stack_detrend.py` и `IO.py`.


In [ ]:
import os
import subprocess
import sys

if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pygmtsar'])
    import importlib.resources as resources
    with resources.as_file(resources.files('pygmtsar.data') / 'google_colab.sh') as script:
        subprocess.check_call(['sh', os.fspath(script)])
    from google.colab import output
    output.enable_custom_widget_manager()


In [ ]:
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client
from shapely.geometry import Polygon

from pygmtsar import ASF, S1, Stack, Tiles

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 140
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)


In [ ]:
END_DATE = pd.Timestamp.utcnow().tz_localize(None).normalize()
START_DATE = END_DATE - pd.DateOffset(years=5)

# AOI - это не одно поле, а часть сцены. Замените полигон на свой рабочий контур.
AOI = gpd.GeoDataFrame(
    geometry=[Polygon([
        (49.95, 55.25),
        (50.32, 55.25),
        (50.32, 55.02),
        (49.95, 55.02),
        (49.95, 55.25),
    ])],
    crs='EPSG:4326',
)

AUTUMN_MONTHS = (9, 10, 11)
BRIDGE_MONTH = 2
ALLOWED_MONTHS = (BRIDGE_MONTH, *AUTUMN_MONTHS)

POLARIZATION = 'VV'
SUBSWATH = 123   # если AOI целиком внутри одного subswath, замените на 1/2/3
WORKDIR = 'raw_tatarstan_scene_sbas'
DATADIR = 'data_tatarstan_scene_sbas'
DEM = f'{DATADIR}/dem.nc'

PAIR_DAYS_AUTUMN = 48
PAIR_DAYS_BRIDGE = 420
PAIR_METERS = 250
INTF_RESOLUTION = 90
INTF_WAVELENGTH = 180
CORR_THRESHOLD = 0.12

print(f'Окно обработки: {START_DATE.date()} — {END_DATE.date()}')
AOI


In [ ]:
def prepare_scene_results(search_gdf):
    gdf = search_gdf.copy()
    gdf['date'] = pd.to_datetime(gdf['startTime']).dt.normalize()
    gdf['month'] = gdf['date'].dt.month
    gdf['year'] = gdf['date'].dt.year
    gdf['orbit_code'] = gdf['flightDirection'].map({
        'ASCENDING': 'A',
        'DESCENDING': 'D',
    }).fillna(gdf['flightDirection'].astype(str).str[0])
    if 'frameNumber' not in gdf.columns:
        gdf['frameNumber'] = -1
    return gdf


def add_overlap_area(search_gdf, aoi):
    gdf = search_gdf.copy()
    aoi_union = aoi.to_crs(3857).geometry.unary_union
    gdf_3857 = gdf.to_crs(3857)
    gdf['overlap_km2'] = gdf_3857.geometry.intersection(aoi_union).area / 1_000_000
    return gdf


def rank_scene_tracks(search_gdf, allowed_months=ALLOWED_MONTHS):
    gdf = prepare_scene_results(search_gdf)
    gdf = gdf[gdf['month'].isin(allowed_months)].copy()
    rows = []
    for (orbit_code, path_number, frame_number), group in gdf.groupby(['orbit_code', 'pathNumber', 'frameNumber']):
        date_table = group[['date', 'month']].drop_duplicates()
        autumn_dates = int(date_table[date_table['month'].isin(AUTUMN_MONTHS)]['date'].nunique())
        feb_dates = int(date_table[date_table['month'] == BRIDGE_MONTH]['date'].nunique())
        total_dates = int(date_table['date'].nunique())
        mean_overlap = float(group['overlap_km2'].mean()) if 'overlap_km2' in group else np.nan
        rows.append({
            'orbit_code': orbit_code,
            'pathNumber': int(path_number),
            'frameNumber': int(frame_number),
            'autumn_dates': autumn_dates,
            'february_dates': feb_dates,
            'total_dates': total_dates,
            'mean_overlap_km2': mean_overlap,
            'score': autumn_dates * 100 + feb_dates * 25 + total_dates,
        })
    return pd.DataFrame(rows).sort_values(
        ['score', 'autumn_dates', 'february_dates', 'mean_overlap_km2'],
        ascending=False,
    ).reset_index(drop=True)


def choose_reference_date(scene_gdf):
    gdf = prepare_scene_results(scene_gdf)
    dates = pd.Series(sorted(gdf['date'].drop_duplicates()))
    preferred = dates[dates.dt.month.isin(AUTUMN_MONTHS)]
    if preferred.empty:
        preferred = dates[dates.dt.month == BRIDGE_MONTH]
    if preferred.empty:
        preferred = dates
    return preferred.iloc[len(preferred) // 2].strftime('%Y-%m-%d')


def select_one_scene_per_date(scene_gdf, aoi):
    gdf = add_overlap_area(prepare_scene_results(scene_gdf), aoi)
    selected = (
        gdf.sort_values(['date', 'overlap_km2', 'fileID'], ascending=[True, False, True])
        .groupby('date')
        .head(1)
        .sort_values('date')
        .reset_index(drop=True)
    )
    return selected


def build_farmland_pairs(sbas, autumn_days=PAIR_DAYS_AUTUMN, bridge_days=PAIR_DAYS_BRIDGE, meters=PAIR_METERS):
    import pandas as pd

    all_pairs = sbas.sbas_pairs(days=bridge_days, meters=meters).copy()

    autumn = all_pairs[
        all_pairs['ref'].dt.month.isin(AUTUMN_MONTHS)
        & all_pairs['rep'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['duration'] <= autumn_days)
    ].copy()

    autumn_to_feb = all_pairs[
        all_pairs['ref'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['rep'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year + 1)
    ].copy()
    autumn_to_feb['bridge_key'] = autumn_to_feb['rep'].dt.year
    autumn_to_feb = autumn_to_feb.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(2)

    feb_to_autumn = all_pairs[
        (all_pairs['ref'].dt.month == BRIDGE_MONTH)
        & all_pairs['rep'].dt.month.isin(AUTUMN_MONTHS)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year)
    ].copy()
    feb_to_autumn['bridge_key'] = feb_to_autumn['ref'].dt.year
    feb_to_autumn = feb_to_autumn.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(2)

    feb_to_feb = all_pairs[
        (all_pairs['ref'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.month == BRIDGE_MONTH)
        & (all_pairs['rep'].dt.year == all_pairs['ref'].dt.year + 1)
    ].copy()
    feb_to_feb['bridge_key'] = feb_to_feb['ref'].dt.year
    feb_to_feb = feb_to_feb.sort_values(['bridge_key', 'duration', 'baseline']).groupby('bridge_key').head(1)

    pairs = pd.concat([autumn, autumn_to_feb, feb_to_autumn, feb_to_feb], ignore_index=True)
    pairs = pairs.drop(columns=['bridge_key'], errors='ignore')
    pairs = pairs.drop_duplicates(subset=['ref', 'rep']).sort_values(['ref', 'rep']).reset_index(drop=True)

    filler = sbas.sbas_pairs_fill(pairs[['ref', 'rep']])
    if filler is not None:
        filler = filler.merge(
            all_pairs.drop(columns=['pair']).copy(),
            on=['ref', 'rep', 'duration'],
            how='left',
        )
        pairs = pd.concat([pairs, filler], ignore_index=True)
        pairs = pairs.drop_duplicates(subset=['ref', 'rep']).sort_values(['ref', 'rep']).reset_index(drop=True)

    assert len(pairs) > 0, 'Не удалось собрать сеть SBAS-пар. Проверьте AOI и состав сцен.'
    return pairs


## 1. Поиск SLC-сцен, а не burst'ов

Здесь используется уровень **SLC**, потому что территория — это **часть сцены**, задаваемая AOI. После выбора трека и дат скачиваются только нужные `SUBSWATH`, а затем стек режется по AOI.

Ниже используются вызовы, сверенные с исходным кодом библиотеки: `ASF.search(...)`, `ASF.download(..., subswaths=...)`, `S1.scan_slc(..., subswath=...)` и дальнейшие методы `Stack`.


In [ ]:
import asf_search

search_all = ASF.search(
    AOI,
    startTime=str(START_DATE.date()),
    stopTime=str(END_DATE.date()),
    flightDirection=None,
    processingLevel=asf_search.PRODUCT_TYPE.SLC,
)
search_all = add_overlap_area(prepare_scene_results(search_all), AOI)
search_all = search_all[search_all['month'].isin(ALLOWED_MONTHS)].copy()

track_ranking = rank_scene_tracks(search_all)
display(track_ranking.head(10))


In [ ]:
best_track = track_ranking.iloc[0]
selected_track = search_all[
    (search_all['orbit_code'] == best_track['orbit_code'])
    & (search_all['pathNumber'] == best_track['pathNumber'])
    & (search_all['frameNumber'] == best_track['frameNumber'])
].copy()
selected_scenes = select_one_scene_per_date(selected_track, AOI)

REFERENCE = choose_reference_date(selected_scenes)
SCENES = selected_scenes['fileID'].tolist()

print(f"Выбран трек: orbit={best_track['orbit_code']}, path={best_track['pathNumber']}, frame={best_track['frameNumber']}")
print(f'Опорная дата: {REFERENCE}')
print(f'Сцен к скачиванию: {len(SCENES)}')
display(selected_scenes[['date', 'month', 'pathNumber', 'frameNumber', 'overlap_km2', 'fileID']].head(30))


In [ ]:
ax = selected_scenes.boundary.plot(figsize=(8, 8), color='steelblue', linewidth=0.8)
AOI.boundary.plot(ax=ax, color='red', linewidth=2)
ax.set_title('Выбранные SLC-сцены и AOI')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()


## 2. Скачивание частичных сцен

`ASF.download()` скачивает не весь архив целиком, а только нужные subswath и поляризацию. Для AOI, покрывающего лишь часть сцены, это правильнее, чем собирать burst-стек вручную.


In [ ]:
asf_username = None
asf_password = None

asf = ASF(asf_username, asf_password)
print(asf.download(DATADIR, SCENES, SUBSWATH, polarization=POLARIZATION))

scenes = S1.scan_slc(DATADIR, subswath=SUBSWATH, polarization=POLARIZATION)
S1.download_orbits(DATADIR, scenes)
Tiles().download_dem(AOI, filename=DEM)


## 3. Инициализация стека и обрезка части сцены по AOI


In [ ]:
if 'client' in globals():
    client.close()
client = Client()
client


In [ ]:
scenes = S1.scan_slc(DATADIR, subswath=SUBSWATH, polarization=POLARIZATION)
sbas = Stack(WORKDIR, drop_if_exists=True).set_scenes(scenes).set_reference(REFERENCE)
display(sbas.to_dataframe().head())
sbas.plot_scenes(AOI=AOI)


In [ ]:
# Сначала работаем на уровне частичных сцен, затем еще сильнее сужаем геометрию AOI.
sbas.compute_reframe(AOI)
sbas.load_dem(DEM, AOI)
sbas.compute_align()
sbas.compute_geocode(1)

sbas.plot_topo(quantile=[0.01, 0.99])


## 4. SBAS-сеть для низкой когерентности на пашне


In [ ]:
pairs = build_farmland_pairs(sbas)
display(pairs[['ref', 'rep', 'duration', 'baseline']].head(30))
print(f'Всего SBAS-пар: {len(pairs)}')
sbas.plot_baseline(pairs)


## 5. Устойчивый SBAS pipeline

Здесь остаются те же идеи, что и раньше, но уже для части сцены:

- PS-веса;
- агрессивный multilooking;
- unwrap только там, где корреляция приемлемая;
- удаление главной атмосферно-топографической компоненты перед инверсией.


In [ ]:
sbas.compute_ps()
psf = sbas.psfunction()
sbas.plot_psfunction(psf, quantile=[0.05, 0.95])


In [ ]:
sbas.compute_interferogram_multilook(
    pairs,
    'intf_scene_mlook',
    weight=psf,
    resolution=INTF_RESOLUTION,
    wavelength=INTF_WAVELENGTH,
    coarsen=(2, 8),
)

ds_sbas = sbas.open_stack('intf_scene_mlook')
intf_sbas = ds_sbas.phase
corr_sbas = ds_sbas.correlation

sbas.plot_interferograms(intf_sbas[:8], caption='Фаза SBAS, [rad]')
sbas.plot_correlations(corr_sbas[:8], caption='Корреляция SBAS')


In [ ]:
unwrap_input = intf_sbas.where(corr_sbas >= CORR_THRESHOLD)
unwrap_weight = corr_sbas.where(corr_sbas >= CORR_THRESHOLD)

unwrap_sbas = sbas.unwrap_snaphu(
    unwrap_input,
    unwrap_weight,
    conncomp=True,
)
unwrap_sbas = sbas.sync_cube(unwrap_sbas, 'unwrap_scene_sbas')
unwrap_sbas = sbas.conncomp_main(unwrap_sbas, 1)

sbas.plot_phases((unwrap_sbas.phase - unwrap_sbas.phase.mean(['y', 'x']))[:8], caption='Unwrap после отбора главной компоненты')


In [ ]:
decimator = sbas.decimator(resolution=30, grid=(1, 1))
topo = decimator(sbas.get_topo())
inc = decimator(sbas.incidence_angle())
yy, xx = xr.broadcast(topo.y, topo.x)

trend_sbas = sbas.regression(
    unwrap_sbas.phase,
    [
        topo,
        topo * yy,
        topo * xx,
        topo ** 2,
        inc,
        yy,
        xx,
        yy * xx,
    ],
    corr_sbas,
)
trend_sbas = sbas.sync_cube(trend_sbas, 'trend_scene_sbas')

sbas.plot_phases(trend_sbas[:8], caption='Оцененный тренд', quantile=[0.01, 0.99])
sbas.plot_phases((unwrap_sbas.phase - trend_sbas)[:8], caption='Фаза после detrend', vmin=-np.pi, vmax=np.pi)


In [ ]:
disp_sbas = sbas.los_displacement_mm(sbas.lstsq(unwrap_sbas.phase - trend_sbas, corr_sbas))
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_scene_sbas')

velocity_sbas = sbas.velocity(disp_sbas)
velocity_sbas = sbas.sync_cube(velocity_sbas, 'velocity_scene_sbas')

sbas.plot_displacements(disp_sbas[:8], caption='Накопленное LOS-смещение, [mm]', quantile=[0.01, 0.99], symmetrical=True)


In [ ]:
velocity_ll = sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry)

zmin, zmax = np.nanquantile(velocity_ll, [0.01, 0.99])
span = max(abs(zmin), abs(zmax))
fig, ax = plt.subplots(figsize=(9, 6))
velocity_ll.plot.imshow(ax=ax, cmap='turbo', vmin=-span, vmax=span)
AOI.boundary.plot(ax=ax, color='black', linewidth=1)
ax.set_title('Скорость LOS, мм/год')
plt.show()

velocity_ll.to_netcdf('tatarstan_scene_velocity.nc')
print('Сохранено: tatarstan_scene_velocity.nc')


## 6. Что крутить в первую очередь

### Если сеть рвется

1. Проверить, что `SUBSWATH` соответствует AOI. Если зона внутри одной полосы, перейти с `123` на `1`, `2` или `3`.
2. Поднять `INTF_RESOLUTION` до `120` м.
3. Поднять `INTF_WAVELENGTH` до `240` м.
4. Ослабить `CORR_THRESHOLD` до `0.10` или `0.08`.
5. Если осенних сцен мало, добавить август в `AUTUMN_MONTHS`.

### Если слишком сглажено

1. Уменьшить `INTF_RESOLUTION` до `60` м.
2. Уменьшить `INTF_WAVELENGTH` до `120` м.
3. Повысить `CORR_THRESHOLD` до `0.15`.
4. Оставить scene-based загрузку, но сузить сам `AOI`, если внутри него слишком неоднородное землепользование.
